#I am becoming an eligible "DATA INGESTION DEVELOPER/ENGINEER"

#### First Let's understand the basic Catalog + Volume Feature of Databricks (We are learning how to build & use Databricks DATALAKE & LAKEHOUSE as a first cut)
Filesystem Hierarchy of Volume in Databricks (DBFS) **DATALAKE**?<br>
Catalog -> /OurWorkspace/catalog/schema(database)/volume/folder/data files<br>
A Volume is used to store and manage files such as CSV, JSON, Parquet, TXT, etc.

```text
Unity Catalog
    |
    +-- Catalog
          |
          +-- Schema (Database)
                |
                +-- Volume (DATALAKE)
                      |
                      +-- Folder
                            |
                            +-- Data Files
                                  |
                                  +-- customers.csv
                                  +-- orders.json
                                  +-- transactions.parquet
```
Tables Hierarchy of Databricks **LAKEHOUSE(TABLE STRUCTURE ON TOP OF DATALAKE)**?<br>
Catalog -> /OurWorkspace/catalog/schema(database)/tables/data(dbfs filesystem/some other filesystems)<br>
```text
Unity Catalog
    |
    +-- Catalog
          |
          +-- Schema (Database)
                |
                +-- Tables (LAKEHOUSE)
                      |
                      +-- customers
                      +-- orders
                      +-- transactions
```

In [0]:
%python
#Python FBP & SQL DP
spark.sql("drop catalog if exists catalog1_dropme cascade")

In [0]:
%sql
drop catalog if exists catalog1_dropme cascade;
create catalog if not exists catalog1_dropme;

In [0]:
%sql
create database if not exists catalog1_dropme.schema1_dropme;

In [0]:
%sql
create volume if not exists catalog1_dropme.schema1_dropme.volume1_dropme;

In [0]:
%sql
create table if not exists catalog1_dropme.schema1_dropme.sample_table_dropme(id int,name varchar(100));
insert into catalog1_dropme.schema1_dropme.sample_table_dropme values(1,'irfan'),(2,'inceptez');

In [0]:
dbutils.fs.mkdirs("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme")
#Upload the drive data into this location...

In [0]:
dbutils.fs.put("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/file1_dropme.txt",'some data',True)

**PLS GO AND ADD DATA INTO THE ABOVE VOLUME CREATED... FOR PLAYING WITH SPARK**

###We fundamentally understood overall how to work in unity catalog wrt VOLUMES(DATALAKE) & TABLES(LAKEHOUSE)

#Spark SQL/DSL - 1.E(Extract) 
L(Load)<br>
Inbuilt libraries sources/targets & Inbuilt data Formats<br>
2. Bread & Butter (T(Transformation) A(Analytical))<br>

In [0]:
print(spark)

In [0]:
#In databricks we don't require this cell since DBC already created spark session object
from pyspark.sql.session import SparkSession
#sparkobj=SparkSession(config)#traditional approach to construct a class as an object (drawback is spark object is created everytime whether it is already available or not)
#spark.stop()
spark1=SparkSession.builder.getOrCreate()
print(spark1)

##Learn How to Create Dataframes from filesystem using different options
Download the data from the below drive url <br>
https://drive.google.com/drive/folders/1Tw7V9eBtUxy0xQMW38z3-bzWI_ewzLm6?usp=drive_link

###How Do We Write a Typical Spark Application (Core(Obsolute),SQL(Important),Streaming(Mid level important))
####Before we Create Dataframe/RDD, what is the prerequisite? We need to create spark session object by instantiating sparksession class (by default databricks did that if you create a notebook)

In [0]:
from pyspark.sql.session import SparkSession
print(spark)#already instantiated by databricks
spark1=SparkSession.builder.getOrCreate()
print(spark1)#we instantiated

Create a DBFS volume namely commondata and upload the above data in that volume
What are other FS uri's available? file:///, hdfs:///, dbfs:///, gs:///, s3:///, adls:///, blob:///

####How to Read/Extract the data from the filesytem and load it into the distributed memory for further processing/load - using diffent methodologies/options from different sources(fs & db) and different builtin formats (csv/json/orc/parquet/delta/tables)

In [0]:
#options - header, inferSchema, sep/delimiter
df1=spark.read.csv("dbfs:/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs_header",inferSchema=True,header=True,sep=',')
print(df1.printSchema())
display(df1.limit(5))

In [0]:

#If I don't use any options in this csv function, what is the default functionality?
#1. By default it will consider ',' as a delimiter (sep='~')
#2. By default it will use _c0,_c1..._cn it will apply as column headers (header=True or toDF("","","") or we have more options like schema to see further)
#3. By default it will treat all columns as string (inferSchema=True or we have more options to see further)
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs")
#csv_df1.show(2)#display with produce output in a dataframe format
print(csv_df1.printSchema())
display(csv_df1)#display with produce output in a beautified table format, specific to databricks
csv_df1.show(10)


csv_df1=spark.read.csv
(path: PathOrPaths, schema: Optional[Union[StructType, str]]=None, sep: Optional[str]=None, encoding: Optional[str]=None, quote: Optional[str]=None, escape: Optional[str]=None, comment: Optional[str]=None, header: Optional[Union[bool, str]]=None, inferSchema: Optional[Union[bool, str]]=None, ignoreLeadingWhiteSpace: Optional[Union[bool, str]]=None, ignoreTrailingWhiteSpace: Optional[Union[bool, str]]=None, nullValue: Optional[str]=None, nanValue: Optional[str]=None, positiveInf: Optional[str]=None, negativeInf: Optional[str]=None, dateFormat: Optional[str]=None, timestampFormat: Optional[str]=None, maxColumns: Optional[Union[int, str]]=None, maxCharsPerColumn: Optional[Union[int, str]]=None, maxMalformedLogPerPartition: Optional[Union[int, str]]=None, mode: Optional[str]=None, columnNameOfCorruptRecord: Optional[str]=None, multiLine: Optional[Union[bool, str]]=None, charToEscapeQuoteEscaping: Optional[str]=None, samplingRatio: Optional[Union[float, str]]=None, enforceSchema: Optional[Union[bool, str]]=None, emptyValue: Optional[str]=None, locale: Optional[str]=None, lineSep: Optional[str]=None, pathGlobFilter: Optional[Union[bool, str]]=None, recursiveFileLookup: Optional[Union[bool, str]]=None, modifiedBefore: Optional[Union[bool, str]]=None, modifiedAfter: Optional[Union[bool, str]]=None, unescapedQuoteHandling: Optional[str]=None) -> "DataFrame"

Sample data of custs_1
4000001,Kristina,Chung,55,Pilot<br>
4000002,Paige,Chen,77,Teacher

Sample data of custs_header<br>
custid,fname,lname,age,profession<br>
4000001,Kristina,Chung,55,Pilot<br>
4000002,Paige,Chen,77,Teacher

In [0]:
csv_df1=spark.read.format("csv").option('header',True).option('inferSchema',False).load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs")
print(csv_df1.printSchema())

In [0]:
#1. Header Concepts (Either we have define the column names or we have to use the column names from the data)
#Important option is...
#By default it will use _c0,_c1..._cn it will apply as column headers, but we are asking spark to take the first row as header and not as a data?
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs_header",header=True)
print(csv_df1.printSchema())
csv_df1.show(2)
#csv_df1.write.csv("/Volumes/workspace/wd36schema2/volume1/folder1/outputdata")
#By default it will use _c0,_c1..._cn it will apply as column headers, if we use toDF(colnames) we can define our own headers..
csv_df2=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs").toDF("id","fname","lname","age","prof")
#csv_df1.show(2)#display with produce output in a dataframe format
print(csv_df2.printSchema())
csv_df2.show(2)

In [0]:
#2. Printing Schema (equivalent to describe table)
csv_df1.printSchema()
csv_df2.printSchema()

In [0]:
#3. Inferring Schema 
# (Performance Consideration: Use this function causiously because it scans the entire data by immediately evaluating and executing
# hence, not good for large data or not good to use on the predefined schema dataset)
#sample data
#4004979,Tara,Drake,32,
#4004980,Earl,Hahn,34,Human resources assistant
#4004981,Don,Jones,THIRTY SIX,Lawyer
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs",inferSchema=True).toDF("id","fname","lname","age","prof")
#csv_df1.where("id in (4004979,4004981)").show(2)
csv_df1.printSchema()

Performance Importance: Though inferSchema has to be used causiously, we can improve performance by using an option to reduce the data scanned for large data..

In [0]:
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs",inferSchema=True,samplingRatio=.10).toDF("id","fname","lname","age","prof")
csv_df1.where("id in (4004979,4004981)").show(2)
csv_df1.printSchema()

In [0]:
#4. Using delimiter or seperator option
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del",header=True,sep='~',inferSchema=True)
csv_df1.show(2)
csv_df1.printSchema()


In [0]:
#5. Using different options to create dataframe with csv and other module... (2 methodologies (spark.read.inbuiltfunction (inline methodology) or spark.read.format(anyfunction).load("path")) with 3 ways of creating dataframes (pass parameters to the csv()/option/options)) or we can use format().load()
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del",inferSchema=True,header=True,sep='~')
csv_df1.show(2)

#or

csv_df1=spark.read.option("header","True").option("inferSchema","true").option("delimiter","~").csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del")
csv_df1.show(2)

#or 

# another way of creating dataframe (from any sources whether builtin or external)...
#option can be used for 1 or 2 option...
csv_df2=spark.read.option("header","True").option("inferSchema","true").option("delimiter","~").format("csv").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del")
csv_df2.show(2)

#options can be used for multiple options in one function as a parameter...
csv_df3=spark.read.format("csv").options(header="True",inferSchema="true",sep="~").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del")
csv_df3.show(2)

#or

#Best for a unified, source-agnostic syntax (easily swapped for bigquery, snowflake, Parquet, JSON, or Delta)
csv_df1=spark.read.format("csv").options(header="True",inferSchema="true",sep="~").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del")
csv_df1.show(2)
csv_df1.printSchema()

### Preferred way of read and load data into dataframe using fundamental options from built in sources (csv/orc/parquet/xml/json/table) (inferschema, header, sep)

In [0]:
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del",inferSchema=True,header=True,sep='~')
csv_df1.show(2)

### Generic way of read and load data into dataframe using extended options from external sources (bigquery/redshift/athena/synapse) (tmpfolder, access controls)
External connectors (bigquery, snowflake, mongodb, delta) rely on third-party packages (spark-bigquery-connector), so PySpark requires .format("bigquery") to route the request to that specific connector.

In [0]:
#options can be used for multiple options in one function as a parameter...
csv_df3=spark.read.options(header="True",inferSchema="true",sep="~").format("csv").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header_oth_del")
csv_df3.show(2)

### Reading data from multiple files & Multiple Path (We still have few more options)

In [0]:
#Reading multiple files in same location with different filenames patters in a single dataframe using wildcard(*) 
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs_header*",inferSchema=True,header=True,sep=',')
print(csv_df1.count())

In [0]:
#Reading multiple files from multiple paths with direct filename and wildcard also...
csv_df1=spark.read.csv(path=["/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/data/custs_header","/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs_header*"],inferSchema=True,header=True,sep=',')
print(csv_df1.count())

####Requirement: I am getting data from different source systems of different regions (NY, TX, CA) into different landing pad (locations), how to access this data?

In [0]:
df_multiple_sources=spark.read.csv(path=["/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/sourcedir/NY","/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/sourcedir/TX/"],inferSchema=True,header=True,sep=',',pathGlobFilter="custs_header_*")
#.toDF("cid","fn","ln","a","p")
print(df_multiple_sources.count())
#GETTING 2 FILES COUNT
#df_multiple_sources.show(2)
#Multi-Source Consolidation (path=[...]), Deep Folder Traversal (recursiveFileLookup=True), File Name Pattern Filtering (pathGlobFilter="custs_header_*")
df_multiple_sources=spark.read.csv(path=["/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/sourcedir/custs_head","/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/sourcedir/NY/","/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/sourcedir/TX/"],inferSchema=True,header=True,sep=',',pathGlobFilter="custs_header_*",recursiveFileLookup=True)
#.toDF("cid","fn","ln","a","p")
print(df_multiple_sources.count())

In [0]:
#inferschema automatically samplingRatio=.20
#Benifit - Auto schema, drawbacks - takes more time to read the data and also it will not work for custom data types.
df=spark.read.csv(path=["/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs"],inferSchema=True).toDF("id","fname","lname","age","prof")
df.printSchema()
display(df)

#alternative for inferschema - to define SIMPLE custom schema & to improve performance
custom_schema_nonfamiliar="id integer,fname string,lname string,age integer,profession string"#simple way but not familiar way, used only for SIMPLE datatypes
df2=spark.read.schema(custom_schema_nonfamiliar).csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs")
df2.printSchema()
display(df2)

#alternative for inferschema - to define SIMPLE/COMPLEX custom schema & to improve performance
#For example:
from pyspark.sql.types import StructType,StructField,StringType,ShortType,LongType
#create table table1 (id int not null,name string) - StructType([StructField("id",IntegerType(),False),StructField("name",StringType(),True)])
schema1=StructType([StructField("id",LongType(),False),StructField("name",StringType(),True)])

from pyspark.sql.types import StructType, StructField, LongType, StringType, ShortType
custom_schema = StructType([StructField("id", LongType(), False),
    StructField("fname", StringType(), True),
    StructField("lname", StringType(), True),
    StructField("age", ShortType(), True),
    StructField("profession", StringType(), True)]) #custom schema we use standardly and it is familiar

df3=spark.read.schema(custom_schema).csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs")
df3.printSchema()
display(df3)


### Provide schema with SQL String or programatically (very very important)
[PySpark SQL Datatypes](https://spark.apache.org/docs/latest/sql-ref-datatypes.html) <br>
[Data Types](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html)<br>
###To provide schema (columname & datatype), what are the 2 basic options available that we learned so far ? inferSchema/toDF<br>
#Inferschema Benifit - Auto schema, drawbacks - takes more time to read the data and also it will not work for custom data types.<br>
**We are going to learn additionally 2 more options to handle schema (colname & datatype)?**<br>
- 1. Using simple string format of define schema (non familiar way, but simple way).<br>
- 2. IMPORTANT: Using structure type to define schema. (familiar and market standard way, where we can go with more options of using complex schema also) acts as an alternative for inferschema - to define SIMPLE/COMPLEX custom schema & to improve performance also<br>


In [0]:
#By default it will use _c0,_c1..._cn it will apply as column headers, if we use toDF(colnames) we can define our own headers..

csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs").toDF("id","fname","lname","age","prof")
print(csv_df1.printSchema())
csv_df1=spark.read.csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs",inferSchema=True).toDF("id","fname","lname","age","prof")
print(csv_df1.printSchema())

#1. Using simple string format of define custom simple schema.
str_struct="id integer,fname string,lname string,age integer,prof string"
csv_df1=spark.read.schema(str_struct).csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs")
print(csv_df1.printSchema())
csv_df1.show(2)

#2. Important part - Using structure type to define custom complex schema.
#4000001,Kristina,Chung,55,Pilot
#pattern - 
#import the types library based classes..
# define_structure=StructType([StructField("colname",DataType(),True),StructField("colname",DataType(),True)...])
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
custom_schema=StructType([StructField("id",IntegerType(),False),StructField("fname",StringType(),True),StructField("lname",StringType(),True),StructField("age",IntegerType(),True),StructField("prof",StringType())])
csv_df1=spark.read.schema(custom_schema).csv("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/dir1_dropme/custs")
print(csv_df1.printSchema())
csv_df1.show(2)

what will happen if one of the file have an extra column, rest all the columns are same?<br>
Is union operation performing here in dataframe? not directly, but we can do to answer above question.. unionByName (Schema evolution)



####5. Reading data from other formats (Try the below usecases after completing the 3-Basic-WriteOps)

In [0]:
#To understand what is json data, lets write our memory csv data in a form of DF into the DBFS fs in a format of JSON/ORC/PARQUET/DELTA/XML...
csv_df1.write.json("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/cust_json/",mode="overwrite")
#DF 4000001,Kristina,Chung,55,Pilot -> {"id":4000001,"fname":"Kristina","lname":"Chung","age":55,"prof":"Pilot"}
csv_df1.write.xml("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/cust_xml/",mode="overwrite",rowTag="cust")
#Orc/parquet/delta/iceberg is a optimized row/column data format intelligent & performant & compressed
csv_df1.write.orc("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/custorc/",mode="overwrite")
csv_df1.write.parquet("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/custparquet/",mode="overwrite")
csv_df1.write.format("delta").save("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/custdelta/",mode="overwrite")
csv_df1.write.saveAsTable("catalog1_dropme.schema1_dropme.custtbl",mode="overwrite")



####2. Reading json data

In [0]:
dfjson1=spark.read.json("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/cust_json/")
dfjson1.show(2)
display(dfjson1.take(2))

####3. Reading xml data

In [0]:
dfxml1=spark.read.xml("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/cust_xml/",rowTag="cust")
dfxml1.show(2)
display(dfxml1.take(3))

####4. Reading serialized data (orc/parquet/delta)

In [0]:
print("orcdata")
dforc1=spark.read.orc("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/custorc/")
dforc1.show(2)
display(dforc1.take(2))
print("parquetdata")
dfparquet1=spark.read.parquet("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/custparquet/")
dfparquet1.show(2)
display(dfparquet1.take(2))
print("deltadata")
dfdelta1=spark.read.format("delta").load("/Volumes/catalog1_dropme/schema1_dropme/volume1_dropme/data/custdelta/")
dfdelta1.show(2)
display(dfdelta1.take(2))

####5. Reading delta (hive) table data (Lakehouse)

In [0]:
dftable1=spark.read.table("catalog1_dropme.schema1_dropme.custtbl")
display(spark.sql("select * from catalog1_dropme.schema1_dropme.custtbl where prof='Pilot'").take(2))
dftable1.show(2)
display(dftable1.take(2))

What are all the overall options we used in this notebook, for learning fundamental spark csv read operations?
1. How to create/manage Catalog & Volume
2. spark session
3. How to upload/import some sample data and push it to the volume/folder/file
4. spark.read.csv operations (with multiple way of writing code) & spark.read.format().load()
5. Few of the important read options under csv such as header, sep, inferSchema, toDF.
6. How to define custom schema/structure using mere string with colname & datatype or by using StructType([StructField(colname,DataType()...)]) (very important).
7. Few additional options such as samplingRatio, recursiveFileLookup & pathGlobFilter for accessing folders and subfolders with some pattern of filenames
8. How to Read data from different sources and different file formats using spark built in functions.
9. Ultimately we are going understand what are the overall spark.read.functions supported in this notebook.
